# Fine-tuning de Qwen3.5-2B con LoRA (16 bits) – Agente de Infraestructura de TI

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.

Pasos: instalar → cargar el modelo → LoRA → dataset → entrenar → probar → exportar a GGUF y subir a Hugging Face.

Por qué **LoRA en 16 bits y no QLoRA**: Unsloth desaconseja QLoRA (4 bits) en Qwen3.5 porque el error de cuantización es mayor de lo normal. El modelo de 2B con LoRA en 16 bits usa ~5 GB de VRAM, así que cabe en la T4 gratuita.

> Si la celda de instalación falla, abre el notebook oficial de Unsloth para Qwen3.5 (2B) y copia su celda de instalación:
> https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install --upgrade "transformers>=5.0.0"   # Qwen3.5 requiere transformers v5

## 1. Sube el dataset
Sube `train.jsonl` y `val.jsonl` (de `entrenamiento/dataset/`) con el ícono de carpeta de la izquierda, o ejecuta la celda y elígelos.

In [ ]:
import os
if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl")):
    from google.colab import files
    files.upload()
print(sum(1 for _ in open("train.jsonl")), "ejemplos de entrenamiento |", sum(1 for _ in open("val.jsonl")), "de validación")

## 2. Carga el modelo base (16 bits)

In [ ]:
from unsloth import FastLanguageModel

MODELO_BASE = "Qwen/Qwen3.5-2B"
MAX_SEQ = 2048   # instrucciones + 4 fichas + respuesta caben de sobra (~1.500 tokens)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODELO_BASE,
    max_seq_length = MAX_SEQ,
    load_in_4bit = False,     # QLoRA desaconsejado en Qwen3.5
    load_in_16bit = True,     # LoRA en 16 bits
    full_finetuning = False,
)

## 3. Adaptador LoRA
r = 16 sobre todas las proyecciones lineales: se entrena menos del 1 % de los parámetros.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ,
)
model.print_trainable_parameters()

## 4. Dataset con la MISMA plantilla de chat que usará llama.cpp
Se usa `enable_thinking=False`, igual que en producción. Revisa el ejemplo impreso: la causa más común de que un modelo exportado funcione peor es una plantilla de chat distinta entre entrenamiento e inferencia.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def a_texto(ej):
    return {"text": tokenizer.apply_chat_template(ej["messages"], tokenize=False,
                                                  add_generation_prompt=False, enable_thinking=False)}

ds = ds.map(a_texto, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print(ds["train"][0]["text"][-900:])

## 5. Entrenamiento (la pérdida solo se calcula sobre la respuesta del asistente)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["validation"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,          # con ~200 ejemplos, más épocas tienden a sobreajustar
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
# Solo la respuesta cuenta en la pérdida: el modelo no gasta capacidad aprendiendo a repetir las fichas
trainer = train_on_responses_only(trainer,
                                  instruction_part = "<|im_start|>user\n",
                                  response_part = "<|im_start|>assistant\n")

# Verificación: lo que queda sin máscara debe ser SOLO la respuesta del asistente
fila = trainer.train_dataset[0]
visible = [t for t, l in zip(fila["input_ids"], fila["labels"]) if l != -100]
print(tokenizer.decode(visible)[:600])

In [ ]:
stats = trainer.train()
import json
json.dump(trainer.state.log_history, open("historial_entrenamiento.json", "w"), indent=1)
print(stats)

Revisa que la pérdida de validación (`eval_loss`) baje y no empiece a subir al final. Si sube, hay sobreajuste: repite con `num_train_epochs = 1`.

## 6. Prueba rápida con una pregunta de validación

In [ ]:
FastLanguageModel.for_inference(model)
ej = load_dataset("json", data_files="val.jsonl")["train"][0]
mensajes = ej["messages"][:2]
entrada = tokenizer.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True,
                                        enable_thinking=False, return_tensors="pt").to("cuda")
salida = model.generate(input_ids=entrada, max_new_tokens=350, temperature=0.7, top_p=0.8, top_k=20)
print(tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True))

## 7. Exporta a GGUF Q4_K_M y súbelo a tu Hugging Face (repositorio privado)
Necesitas un token de **escritura** de Hugging Face. Tarda varios minutos (compila llama.cpp y convierte).

In [ ]:
from getpass import getpass
from huggingface_hub import create_repo

HF_USUARIO = input("Tu usuario de Hugging Face: ").strip()
HF_TOKEN = getpass("Token de escritura de Hugging Face: ")
REPO_GGUF = f"{HF_USUARIO}/agente-infra-qwen35-2b-gguf"
REPO_LORA = f"{HF_USUARIO}/agente-infra-qwen35-2b-lora"

for repo in (REPO_GGUF, REPO_LORA):
    create_repo(repo, private=True, exist_ok=True, token=HF_TOKEN)

model.push_to_hub(REPO_LORA, token=HF_TOKEN)          # el adaptador LoRA (pocos MB): para reproducibilidad
tokenizer.push_to_hub(REPO_LORA, token=HF_TOKEN)
model.push_to_hub_gguf(REPO_GGUF, tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("Listo:", REPO_GGUF)

In [ ]:
from google.colab import files
files.download("historial_entrenamiento.json")   # evidencia de la curva de pérdida para el reporte